In [ ]:
# Edit only attached Kaggle Input paths and the declared offline revision if known.
from pathlib import Path

BM25_WHEEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/offline-packages/bm25s-0.3.11-py3-none-any.whl"
)
LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
DENSE_MODEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/bge-m3-kaggle"
)

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
BM25_METHOD = "lucene"
BM25_K1 = 1.5
BM25_B = 0.75
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2"
EVALUATION_DEPTHS = (10, 20, 50, 100, 200)
MAX_LENGTH = 8_192
CORPUS_BATCH_SIZE = 256
QUERY_BATCH_SIZE = 64
EXPECTED_DEV_QUERIES = 1_036
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816
RESULT_PATH = Path("/kaggle/working/bge_m3_dense_retrieval_dev_results.json")


In [ ]:
# Fail before imports/model loading if an offline artifact is missing.
import os
import subprocess
import sys

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if not BM25_WHEEL_PATH.is_file():
    raise FileNotFoundError(f"Attach the local BM25 wheel at: {BM25_WHEEL_PATH}")
if not LEGALIR_SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Attach LegalIR train.json at: {LEGALIR_SOURCE_PATH}")
if not CORPUS_PATH.is_dir():
    raise FileNotFoundError(f"Attach the LegalIR corpus directory at: {CORPUS_PATH}")
if not DENSE_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        f"Attach the complete local BGE-M3 embedding snapshot at: {DENSE_MODEL_PATH}"
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        str(BM25_WHEEL_PATH),
    ],
    check=True,
)


In [ ]:
import json
import re
from collections import defaultdict
from hashlib import sha256
from math import isfinite
from time import perf_counter

import bm25s
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_legalir(path: Path) -> dict:
    value = read_json(path)
    if not isinstance(value, dict) or not all(
        isinstance(sample, dict) for sample in value.values()
    ):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    canonical = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(canonical) != len(value):
        raise ValueError(f"{path}: duplicate sample IDs after string canonicalization")
    return canonical


def select_fixed_dev(samples: dict) -> dict:
    dev_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = (
            question
            if isinstance(question, str)
            else f"\0fallback-sample-id:{sample_id}"
        )
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        if 70 <= bucket < 85:
            dev_ids.append(sample_id)
    dev_ids.sort()
    dev = {sample_id: samples[sample_id] for sample_id in dev_ids}
    if len(dev) != EXPECTED_DEV_QUERIES:
        raise ValueError(
            f"fixed DEV must contain {EXPECTED_DEV_QUERIES} queries, got {len(dev)}"
        )
    for sample_id, sample in dev.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"DEV sample {sample_id!r}: question must be a string")
        if not isinstance(sample.get("answer"), list) or not sample["answer"]:
            raise ValueError(
                f"DEV sample {sample_id!r}: expected a non-empty LegalIR answer list"
            )
    return dev


def load_corpus(path: Path) -> list[dict]:
    json_paths = sorted(
        item for item in path.rglob("*") if item.is_file() and item.suffix.lower() == ".json"
    )
    if not json_paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in json_paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = [str(document.get("id")) for document in documents]
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    return documents


def chunk_corpus(documents: list[dict]) -> list[dict]:
    if CHUNK_SIZE <= 0 or CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
        raise ValueError("invalid fixed-window chunk parameters")
    step = CHUNK_SIZE - CHUNK_OVERLAP
    chunks = []
    for document in documents:
        document_id = str(document["id"])
        passage = document.get("passage")
        if not isinstance(passage, str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        if not passage:
            continue
        for chunk_index, start in enumerate(range(0, len(passage), step)):
            end = min(start + CHUNK_SIZE, len(passage))
            chunks.append(
                {
                    "chunk_id": f"{document_id}:{chunk_index}",
                    "document_id": document_id,
                    "text": passage[start:end],
                }
            )
            if end == len(passage):
                break
    return chunks


def aggregate_sum_top_2(chunk_hits: list[dict]) -> list[str]:
    grouped = defaultdict(list)
    best_chunk_rank = {}
    for hit in chunk_hits:
        document_id = hit["document_id"]
        score = float(hit["score"])
        if not isfinite(score):
            raise ValueError("retrieval score must be finite")
        grouped[document_id].append(score)
        best_chunk_rank[document_id] = min(
            best_chunk_rank.get(document_id, hit["rank"]), hit["rank"]
        )
    ranked = [
        (
            document_id,
            sum(sorted(scores, reverse=True)[:2]),
            best_chunk_rank[document_id],
        )
        for document_id, scores in grouped.items()
    ]
    ranked.sort(key=lambda item: (-item[1], item[2], item[0]))
    document_ids = [item[0] for item in ranked]
    if len(document_ids) != len(set(document_ids)):
        raise RuntimeError("document aggregation produced duplicate IDs")
    return document_ids


TOKEN_PATTERN = re.compile(r"\w+", flags=re.UNICODE)


def lexical_tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


def build_bm25_rankings(chunks: list[dict], samples: dict) -> dict:
    if bm25s.__version__ != "0.3.11":
        raise RuntimeError(f"expected bm25s==0.3.11, got {bm25s.__version__}")
    started = perf_counter()
    tokenized = bm25s.tokenize(
        [chunk["text"] for chunk in chunks],
        lower=True,
        token_pattern=r"(?u)\w+",
        stopwords=[],
        stemmer=None,
        return_ids=True,
        show_progress=False,
    )
    retriever = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD)
    retriever.index(tokenized, show_progress=False)

    rankings = {}
    sample_items = list(samples.items())
    for batch_start in range(0, len(sample_items), QUERY_BATCH_SIZE):
        batch = sample_items[batch_start : batch_start + QUERY_BATCH_SIZE]
        result = retriever.retrieve(
            [lexical_tokenize(sample["question"]) for _, sample in batch],
            k=TOP_K_CHUNKS,
            sorted=True,
            return_as="tuple",
            show_progress=False,
        )
        for (sample_id, _), hit_indices, hit_scores in zip(
            batch, result.documents, result.scores
        ):
            hits = []
            for rank, (index_value, score_value) in enumerate(
                zip(hit_indices, hit_scores), start=1
            ):
                chunk = chunks[int(index_value)]
                hits.append(
                    {
                        "document_id": chunk["document_id"],
                        "score": float(score_value),
                        "rank": rank,
                    }
                )
            rankings[sample_id] = aggregate_sum_top_2(hits)
    return {"rankings": rankings, "seconds": perf_counter() - started}


def local_model_metadata(model):
    config_commit_hash = getattr(model.config, "_commit_hash", None)
    verified = isinstance(config_commit_hash, str) and bool(config_commit_hash.strip())
    return {
        "model_name": DENSE_MODEL_NAME,
        "local_input_path": str(DENSE_MODEL_PATH),
        "declared_revision": DENSE_DECLARED_REVISION,
        "config_commit_hash": config_commit_hash if verified else None,
        "revision_status": (
            "verified-from-config" if verified else "declared-offline-snapshot"
        ),
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("This dense coverage notebook requires a Kaggle CUDA accelerator")
    torch.cuda.reset_peak_memory_stats()
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        DENSE_MODEL_PATH,
        local_files_only=True,
    )
    model = AutoModel.from_pretrained(
        DENSE_MODEL_PATH,
        dtype=torch.float16,
        local_files_only=True,
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer,
        "model": model,
        "device": "cuda",
        "dtype": "float16",
        "load_seconds": perf_counter() - started,
        "metadata": local_model_metadata(model),
    }


def encode_normalized_cls(
    dense_model: dict,
    texts: list[str],
    batch_size: int,
    collect_token_lengths: bool,
) -> dict:
    if batch_size <= 0:
        raise ValueError("encoding batch size must be positive")
    tokenizer = dense_model["tokenizer"]
    model = dense_model["model"]
    embeddings = []
    token_lengths = []
    started = perf_counter()

    for batch_start in range(0, len(texts), batch_size):
        batch = texts[batch_start : batch_start + batch_size]
        if collect_token_lengths:
            untruncated = tokenizer(
                batch,
                padding=False,
                truncation=False,
                add_special_tokens=True,
                return_length=True,
            )
            token_lengths.extend(int(length) for length in untruncated["length"])
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, return_dict=True)
            embedding = outputs.last_hidden_state[:, 0]
            embedding = F.normalize(embedding, p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid normalized CLS embeddings")
        embeddings.append(embedding.cpu())

    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense encoder output count does not match input count")
    diagnostics = None
    if collect_token_lengths:
        lengths = np.asarray(token_lengths, dtype=np.int32)
        truncated_count = int(np.sum(lengths > MAX_LENGTH))
        diagnostics = {
            "median": float(np.median(lengths)),
            "p95": float(np.percentile(lengths, 95)),
            "max": int(lengths.max()),
            "truncated_count": truncated_count,
            "truncated_fraction": truncated_count / len(lengths),
        }
    seconds = perf_counter() - started
    return {
        "embeddings": encoded,
        "seconds": seconds,
        "items_per_second": len(texts) / seconds,
        "token_lengths": diagnostics,
    }


def dense_rankings(
    query_embeddings: torch.Tensor,
    corpus_embeddings: torch.Tensor,
    chunks: list[dict],
    sample_ids: list[str],
) -> dict:
    if query_embeddings.shape[0] != len(sample_ids):
        raise ValueError("query embedding count does not match sample IDs")
    if corpus_embeddings.shape[0] != len(chunks):
        raise ValueError("corpus embedding count does not match chunks")
    started = perf_counter()
    passage_embedding = corpus_embeddings.to("cuda")
    rankings = {}

    for batch_start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[batch_start : batch_start + QUERY_BATCH_SIZE]
        query_embedding = query_embeddings[
            batch_start : batch_start + len(batch_ids)
        ].to("cuda")
        score = query_embedding @ passage_embedding.T
        if not torch.isfinite(score).all():
            raise RuntimeError("dense dot-product similarity produced non-finite scores")
        top_scores, top_indices = torch.topk(
            score,
            k=TOP_K_CHUNKS,
            dim=1,
            largest=True,
            sorted=True,
        )
        for row, sample_id in enumerate(batch_ids):
            raw_hits = [
                (float(score_value), int(index_value))
                for score_value, index_value in zip(
                    top_scores[row].float().cpu().tolist(),
                    top_indices[row].cpu().tolist(),
                )
            ]
            raw_hits.sort(key=lambda item: (-item[0], item[1]))
            hits = [
                {
                    "document_id": chunks[index_value]["document_id"],
                    "score": score_value,
                    "rank": rank,
                }
                for rank, (score_value, index_value) in enumerate(raw_hits, start=1)
            ]
            rankings[sample_id] = aggregate_sum_top_2(hits)

    seconds = perf_counter() - started
    return {
        "rankings": rankings,
        "seconds": seconds,
        "queries_per_second": len(sample_ids) / seconds,
    }


def coverage_metrics(samples: dict, rankings: dict) -> dict:
    recall_values = {depth: [] for depth in EVALUATION_DEPTHS}
    reciprocal_ranks = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        ranked = rankings[sample_id]
        if len(ranked) != len(set(ranked)):
            raise ValueError(f"sample {sample_id!r}: ranking contains duplicate IDs")
        for depth in EVALUATION_DEPTHS:
            recall_values[depth].append(len(gold.intersection(ranked[:depth])) / len(gold))
        first_rank = next(
            (rank for rank, document_id in enumerate(ranked, start=1) if document_id in gold),
            None,
        )
        reciprocal_ranks.append(0.0 if first_rank is None else 1.0 / first_rank)

    return {
        "recall": {
            str(depth): {
                "mean": float(np.mean(values)),
                "zero_recall_rate": float(np.mean(np.asarray(values) == 0)),
                "full_recall_rate": float(np.mean(np.asarray(values) == 1)),
            }
            for depth, values in recall_values.items()
        },
        "mrr": float(np.mean(reciprocal_ranks)),
    }


def complementarity(samples: dict, bm25_rankings: dict, dense_results: dict) -> dict:
    output = {}
    for depth in EVALUATION_DEPTHS:
        categories = {
            "gold_found_by_both": 0,
            "gold_found_by_bm25_only": 0,
            "gold_found_by_dense_only": 0,
            "gold_found_by_neither": 0,
        }
        dense_unique_queries = 0
        bm25_unique_queries = 0
        union_recalls = []
        bm25_recalls = []
        dense_recalls = []

        for sample_id, sample in samples.items():
            gold = {str(document_id) for document_id in sample["answer"]}
            bm25_prefix = set(bm25_rankings[sample_id][:depth])
            dense_prefix = set(dense_results[sample_id][:depth])
            categories["gold_found_by_both"] += len(gold & bm25_prefix & dense_prefix)
            categories["gold_found_by_bm25_only"] += len(
                (gold & bm25_prefix) - dense_prefix
            )
            categories["gold_found_by_dense_only"] += len(
                (gold & dense_prefix) - bm25_prefix
            )
            categories["gold_found_by_neither"] += len(
                gold - (bm25_prefix | dense_prefix)
            )
            dense_unique_queries += bool((gold & dense_prefix) - bm25_prefix)
            bm25_unique_queries += bool((gold & bm25_prefix) - dense_prefix)
            bm25_recalls.append(len(gold & bm25_prefix) / len(gold))
            dense_recalls.append(len(gold & dense_prefix) / len(gold))
            union_recalls.append(len(gold & (bm25_prefix | dense_prefix)) / len(gold))

        output[str(depth)] = {
            **categories,
            "queries_where_dense_recovers_at_least_one_gold_not_in_bm25_prefix": int(
                dense_unique_queries
            ),
            "queries_where_bm25_recovers_at_least_one_gold_not_in_dense_prefix": int(
                bm25_unique_queries
            ),
            "bm25_recall": float(np.mean(bm25_recalls)),
            "dense_recall": float(np.mean(dense_recalls)),
            "union_recall_ceiling": float(np.mean(union_recalls)),
        }
    return output


In [ ]:
# Offline model preflight and synthetic embedding smoke check.
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
assert TOP_K_CHUNKS == 2_000
assert DOCUMENT_AGGREGATION == "sum_top_2"
assert MAX_LENGTH == 8_192
assert EVALUATION_DEPTHS == (10, 20, 50, 100, 200)

dense_model = load_dense_model()
smoke = encode_normalized_cls(
    dense_model,
    ["Câu hỏi kiểm tra.", "Đoạn văn kiểm tra."],
    batch_size=2,
    collect_token_lengths=True,
)
assert smoke["embeddings"].shape[0] == 2
assert torch.isfinite(smoke["embeddings"]).all()
assert torch.allclose(
    torch.linalg.vector_norm(smoke["embeddings"].float(), dim=1),
    torch.ones(2),
    atol=1e-3,
)
smoke_score = smoke["embeddings"][0] @ smoke["embeddings"][1].T
assert smoke_score.ndim == 0 and torch.isfinite(smoke_score)
print("Offline preflight and normalized-CLS smoke check passed.")


In [ ]:
run_started = perf_counter()
source_samples = load_legalir(LEGALIR_SOURCE_PATH)
dev_samples = select_fixed_dev(source_samples)
documents = load_corpus(CORPUS_PATH)
chunks = chunk_corpus(documents)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_CHUNKS:
    raise ValueError(
        f"expected {EXPECTED_DOCUMENTS:,} documents / {EXPECTED_CHUNKS:,} chunks, "
        f"got {len(documents):,} / {len(chunks):,}"
    )

bm25 = build_bm25_rankings(chunks, dev_samples)

corpus_encoding = encode_normalized_cls(
    dense_model,
    [chunk["text"] for chunk in chunks],
    batch_size=CORPUS_BATCH_SIZE,
    collect_token_lengths=True,
)
query_encoding = encode_normalized_cls(
    dense_model,
    [sample["question"] for sample in dev_samples.values()],
    batch_size=QUERY_BATCH_SIZE,
    collect_token_lengths=False,
)
dense = dense_rankings(
    query_encoding["embeddings"],
    corpus_encoding["embeddings"],
    chunks,
    list(dev_samples),
)

assert set(bm25["rankings"]) == set(dev_samples)
assert set(dense["rankings"]) == set(dev_samples)
assert all(
    len(ranking) == len(set(ranking))
    for ranking in bm25["rankings"].values()
)
assert all(
    len(ranking) == len(set(ranking))
    for ranking in dense["rankings"].values()
)

bm25_metrics = coverage_metrics(dev_samples, bm25["rankings"])
dense_metrics = coverage_metrics(dev_samples, dense["rankings"])
complement = complementarity(
    dev_samples,
    bm25["rankings"],
    dense["rankings"],
)

dense_only_total = sum(
    values["gold_found_by_dense_only"] for values in complement.values()
)
union_gain_exists = any(
    values["union_recall_ceiling"] > values["bm25_recall"]
    for values in complement.values()
)
if dense_only_total > 0 and union_gain_exists:
    interpretation = (
        "Dense retrieval recovers unique relevant documents and provides "
        "complementary candidate evidence; no hybrid method is selected here."
    )
else:
    interpretation = (
        "No aggregate complementarity was demonstrated under this representation; "
        "the dense candidate pool is largely redundant in this run."
    )

result = {
    "split": {
        "name": "fixed DEV",
        "queries": len(dev_samples),
        "source_sha256": sha256(LEGALIR_SOURCE_PATH.read_bytes()).hexdigest(),
    },
    "controls": {
        "documents": len(documents),
        "chunks": len(chunks),
        "chunk_size": CHUNK_SIZE,
        "overlap": CHUNK_OVERLAP,
        "top_k_chunks": TOP_K_CHUNKS,
        "document_aggregation": "sum top-2 retrieval chunk scores",
        "evaluation_depths": list(EVALUATION_DEPTHS),
        "no_query_instruction": True,
        "no_cross_encoder": True,
        "no_fusion": True,
    },
    "dense_model": {
        **dense_model["metadata"],
        "implementation": "transformers AutoModel normalized CLS hidden state",
        "similarity": "query_embedding @ passage_embedding.T",
        "max_length": MAX_LENGTH,
        "dynamic_padding": True,
    },
    "lexical_reference": {
        "library": f"bm25s=={bm25s.__version__}",
        "method": BM25_METHOD,
        "k1": BM25_K1,
        "b": BM25_B,
        "tokenization": r"lowercase Unicode \w+",
        "metrics": bm25_metrics,
    },
    "dense": {
        "metrics": dense_metrics,
        "number_of_corpus_embeddings": int(corpus_encoding["embeddings"].shape[0]),
        "embedding_dimension": int(corpus_encoding["embeddings"].shape[1]),
        "embedding_dtype": str(corpus_encoding["embeddings"].dtype).removeprefix("torch."),
        "corpus_token_length_before_truncation": corpus_encoding["token_lengths"],
    },
    "complementarity": complement,
    "union_is_coverage_ceiling_only": True,
    "interpretation": interpretation,
    "runtime": {
        "model_load_seconds": dense_model["load_seconds"],
        "bm25_build_and_retrieval_seconds": bm25["seconds"],
        "corpus_encoding_seconds": corpus_encoding["seconds"],
        "query_encoding_seconds": query_encoding["seconds"],
        "retrieval_seconds": dense["seconds"],
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "chunks_per_second_encoding": corpus_encoding["items_per_second"],
        "queries_per_second_retrieval": dense["queries_per_second"],
        "corpus_batch_size": CORPUS_BATCH_SIZE,
        "query_batch_size": QUERY_BATCH_SIZE,
        "total_seconds": perf_counter() - run_started,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
}

RESULT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps({
    "lexical_reference": bm25_metrics,
    "dense": dense_metrics,
    "complementarity": complement,
    "interpretation": interpretation,
    "runtime": result["runtime"],
}, ensure_ascii=False, indent=2))
print("Saved:", RESULT_PATH)
